# pyfixest + MLflow tracking example

This notebook makes some synthetic data, runs it through `regress` (which wraps a pyfixest modeling function and logs to MLflow), and then pulls the logged runs back out to inspect them.

In [ ]:
import warnings

# keep outputs reproducible: this warning embeds an absolute path
warnings.filterwarnings("ignore", message="IProgress not found")

import logging
import os

# artifact-download progress bars print throughput rates -> not reproducible
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"

import mlflow
import numpy as np
import pandas as pd
from IPython.display import Markdown

from pyfixest_regression import coefficients_table, etable, regress, results_table

# mlflow's INFO logs carry timestamps; silence them so outputs are stable
logging.getLogger("mlflow").setLevel(logging.ERROR)

mlflow.set_tracking_uri("sqlite:///mlflow.db")
# Set the experiment once. regress then logs into the active experiment
# (you could also pass experiment_name= per call).
_ = mlflow.set_experiment("pyfixest-example")

## Make some data

A simple linear DGP: `Y = 2 + 1.5*X1 - 0.8*X2 + noise`.

In [ ]:
rng = np.random.default_rng(0)
n = 500

X1 = rng.normal(size=n)
X2 = rng.normal(size=n)
Y = 2.0 + 1.5 * X1 - 0.8 * X2 + rng.normal(scale=1.0, size=n)

data = pd.DataFrame({"Y": Y, "X1": X1, "X2": X2})
data.head()

,Y,X1,X2
0,2.338183,0.125730,1.292893
1,1.742733,-0.132105,0.453671
2,4.504816,0.640423,-1.690160
3,3.005850,0.104900,-0.728195
4,-1.155505,-0.535669,1.232303


## Run it

Two runs on the same data: one with `iid` errors, one with `hetero` (heteroskedasticity-robust) errors. The `vcov` is part of each run's content hash, so these are two distinct runs rather than one deduplicated run. Both calls log into the experiment set above; `name` (the optional run descriptor) is omitted, since runs are identified by their content anyway.

In [ ]:
fit_iid = regress("Y ~ X1 + X2", data=data, vcov="iid")
fit_hetero = regress("Y ~ X1 + X2", data=data, vcov="hetero")

fit_iid.summary()

###

Estimation:  OLS
Dep. var.: Y
sample: None = all
Inference:  iid
Observations:  500

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Intercept     |      2.049 |        0.045 |    45.121 |      0.000 |  1.960 |   2.139 |
| X1            |      1.525 |        0.045 |    34.144 |      0.000 |  1.438 |   1.613 |
| X2            |     -0.827 |        0.048 |   -17.128 |      0.000 | -0.922 |  -0.732 |
---
RMSE: 1.01 R2: 0.745 


## Get the runs data and show it

`regress` returns the fitted model directly, but everything it logged is also in MLflow. `results_table` wraps `mlflow.search_runs` and returns a tidy one-row-per-run frame (params and metrics, prefixes stripped).

In [ ]:
runs = results_table("pyfixest-example")
runs[["fml", "vcov", "nobs", "r2", "adj_r2"]]

,fml,vcov,nobs,r2,adj_r2
0,Y ~ X1 + X2,hetero,500.0,0.74513,0.744105
1,Y ~ X1 + X2,iid,500.0,0.74513,0.744105


## Artifacts logged per run

Each run also logs the coefficient table (`coefficients.json`) and a human-readable regression table (`summary.md`), built from the run's own logged info.

In [ ]:
run_id = runs.loc[0, "run_id"]
[a.path for a in mlflow.artifacts.list_artifacts(run_id=run_id)]

['coefficients.json', 'summary.md']

## Render the logged regression table

The `summary.md` artifact is a self-built markdown regression table for that run. Load it back and render it inline.

In [ ]:
Markdown(mlflow.artifacts.load_text(f"runs:/{run_id}/summary.md"))

### Y ~ X1 + X2

| Coefficient | Estimate | Std. Error | p-value |
|:---|---:|---:|---:|
| Intercept | 2.049*** | 0.045 | 0.000 |
| X1 | 1.525*** | 0.045 | 0.000 |
| X2 | -0.827*** | 0.047 | 0.000 |

| Statistic | Value |
|:---|---:|
| nobs | 500 |
| r2 | 0.745 |
| adj_r2 | 0.744 |
| f_statistic | 1128.704 |
| rmse | 1.010 |

Significance: `*` p < 0.05, `**` p < 0.01, `***` p < 0.001. Cells: estimate with stars; standard error and p-value alongside.

## Coefficients across runs

`coefficients_table` reads each run's `coefficients.json` back into one long, self-describing frame (one row per run × coefficient, with the run's params joined on).

In [ ]:
coefs = coefficients_table("pyfixest-example")
coefs[["Coefficient", "Estimate", "Std. Error", "Pr(>|t|)", "vcov"]]

,Coefficient,Estimate,Std. Error,Pr(>|t|),vcov
0,Intercept,2.049468,0.045186,0,hetero
1,X1,1.525423,0.044730,0,hetero
2,X2,-0.826900,0.047373,0,hetero
3,Intercept,2.049468,0.045422,0,iid
4,X1,1.525423,0.044676,0,iid
5,X2,-0.826900,0.048278,0,iid


Filter to a single coefficient to compare it across runs (here, the slope on `X1` under iid vs hetero errors):

In [ ]:
x1 = coefficients_table("pyfixest-example", coefficients="X1")
x1[["Coefficient", "Estimate", "Std. Error", "Pr(>|t|)", "vcov"]]

,Coefficient,Estimate,Std. Error,Pr(>|t|),vcov
0,X1,1.525423,0.044730,0,hetero
1,X1,1.525423,0.044676,0,iid


## Cross-run regression table

`etable` rebuilds a side-by-side comparison table -- one column per run -- entirely from the logged runs (coefficients, params, metrics). `type="md"` returns it as markdown.

In [ ]:
etable("pyfixest-example")

,(1),(2)
Intercept,2.049*** (0.045),2.049*** (0.045)
X1,1.525*** (0.045),1.525*** (0.045)
X2,-0.827*** (0.048),-0.827*** (0.047)
fml,Y ~ X1 + X2,Y ~ X1 + X2
vcov,iid,hetero
nobs,500,500
r2,0.745,0.745
adj_r2,0.744,0.744
